# Lecture 3/ GV: Additional databases and downloads

## Tasks for today's lecture:
* Continue working with the databases

* Define and download option-based characteristics from OptionMetrics 
* Think about matching the sample with CRSP-based returns 


**Some useful info:** 
* Linearmodels: https://bashtage.github.io/linearmodels/index.html 
* Statmodels: https://www.statsmodels.org/stable/index.html
* Scipy/ scikit-learn: https://www.scipy.org/index.html + https://scikit-learn.org/stable/ 

In [3]:
# after we install all the packages, import all of them for the use in today's lecture!
import platform
my_system = platform.uname()
print(f'My PC node: {my_system.node.lower()}')

# database access
import pandas_datareader as web
import nasdaqdatalink
import wrds as wrds

# storage and operations
import pandas as pd
import numpy as np
import datetime
import time 
from pathlib import Path
import joblib

from statsmodels.regression.rolling import RollingOLS

from joblib import Parallel, delayed
from multiprocessing import Pool, cpu_count
if cpu_count() > 30:
    CPUUsed = 15
else:
    CPUUsed = cpu_count() - 2

DATA_FROM_LIMIT = datetime.datetime(2014,1,1)
# we can specify some options depending on your computer
class Options:
    if (my_system.node.lower() in ['mac','macbookpro-gv.local']): # GREG'S MACHINE - do not change please
        path = Path('../DataQT2026')
        wrds_name = 'gvfs'
        nasdaqlink_key = 'XXX'
    elif (my_system.node.lower() == 'YOU CAN PUT HERE YOUR NODE'):
        path = Path(r'/.../DataQT2026/')
        wrds_name = 'XXX'
        nasdaqlink_key = 'XXX'
        pass
print('Data Path: ',Options.path)

# some file names
class FileNames:
    fn_csv_factors   = Options.path / 'ff_fact.csv'
    fn_excel_factors = Options.path / 'ff_fact.xls'

    fn_crsp     = Options.path / 'crsp.parquet'
    fn_stock_features_labels = Options.path / 'stock_features.parquet'
    fn_option_features = Options.path / 'option_features.parquet'
    fn_ff_factors    = Options.path / 'ff_factors.parquet'

    fn_sp500comp     = Options.path / 'SP500_Index_Constitutes2024.csv'
    fn_id_link = Options.path / 'crsp_secid_link.csv'
    fn_universe = Options.path / 'permno_selection.csv'


My PC node: mac
Data Path:  ../DataQT2026


# Check the data we have already prepared
* Data for stocks: returns, market cap, etc. 
* Linking tables: CRSP/Compustat/OptionMetrics, etc.

In [5]:
'''DEFINE FUNCTIONS TO LOAD THE DATA'''   
def load_ff_crsp():
    crsp = pd.read_parquet(FileNames.fn_crsp)
    ff = pd.read_parquet(FileNames.fn_ff_factors)
    return crsp, ff   


def load_features():
    features_crsp = pd.read_parquet(FileNames.fn_stock_features_labels)
    return features_crsp  

In [6]:
crsp, ff = load_ff_crsp()
features_crsp = load_features()
features_crsp.head()

ret        mktcap    fret1d    fret5d    mom12m  \
permno date                                                               
10104  2014-12-31 -0.008161  200915.32336 -0.014232 -0.032033  0.122049   
       2015-01-02 -0.014232  197479.77399 -0.013986 -0.018511  0.113319   
       2015-01-05 -0.013986  194669.29911 -0.010324 -0.007343  0.115964   
       2015-01-06 -0.010324  191419.68753  0.000232 -0.004869  0.112789   
       2015-01-07  0.000232  189443.57238  0.006025 -0.002087  0.108291   

                      mom6m     rev1m  beta_mktrf  beta_smb  beta_hml  \
permno date                                                             
10104  2014-12-31  0.016560  0.068678    1.141866 -0.269630 -0.541900   
       2015-01-02  0.012509  0.050972    1.140102 -0.258900 -0.539181   
       2015-01-05  0.017593  0.039230    1.123524 -0.252731 -0.542466   
       2015-01-06  0.005057  0.032675    1.124913 -0.251336 -0.540608   
       2015-01-07 -0.011450  0.031929    1.116593 -0.245790 -0.519419   

                   beta_mom  idvar_ff4  
permno date                             
10104  2014-12-31 -0.223871   0.024360  
       2015-01-02 -0.221639   0.024561  
       2015-01-05 -0.200264   0.024600  
       2015-01-06 -0.201985   0.024602  
       2015-01-07 -0.212445   0.024808

# A quick chat about return predictability 
* Time-series predictability
* Cross-sectional predictability 
* Uses of both and testing both

# WRDS: OptionMetrics

In [7]:
db = wrds.Connection(wrds_username=Options.wrds_name)

Loading library list...
Done


In [8]:
# create the list of SECIDs corresponding to our PERMNO universe
# read the sp500 composition and make the string from PERMNO for SQL
# read the sp500 composition and make the string from PERMNO for SQL
comp = pd.read_csv(FileNames.fn_sp500comp)
comp['start']  = pd.to_datetime(comp['start'])
comp['ending'] = pd.to_datetime(comp['ending'])
comp = comp.loc[comp.ending>DATA_FROM_LIMIT,:] 

print(comp.head(),'\n')
np.unique(comp.permno).shape

universe = list(pd.read_csv(FileNames.fn_universe)['permno'])

print(universe,'\n')

crsp_secid_link = pd.read_csv(FileNames.fn_id_link)
crsp_secid_link.columns = [z.lower().strip() for z in crsp_secid_link.columns]
crsp_secid_link['sdate'] = pd.to_datetime(crsp_secid_link['sdate'])
crsp_secid_link['edate'] = pd.to_datetime(crsp_secid_link['edate'])
crsp_secid_link = crsp_secid_link.loc[crsp_secid_link['edate']>DATA_FROM_LIMIT,:]
print(crsp_secid_link)

    permno      start     ending
6    10104 1989-08-03 2024-12-31
7    10107 1994-06-07 2024-12-31
11   10138 1999-10-13 2024-12-31
12   10145 1925-12-31 2024-12-31
13   10147 1996-03-28 2016-09-07 

[10874.0, 11308.0, 11404.0, 11786.0, 13856.0, 16432.0, 16600.0, 17144.0, 17750.0, 18163.0, 18411.0, 18729.0, 20482.0, 21207.0, 21776.0, 22111.0, 22752.0, 24010.0, 24109.0, 24205.0, 25320.0, 26825.0, 27959.0, 42200.0, 43449.0, 45911.0, 46578.0, 47466.0, 47626.0, 48486.0, 51369.0, 52476.0, 53065.0, 53613.0, 56274.0, 61241.0, 64936.0, 69032.0, 71298.0, 75175.0, 75341.0, 76644.0, 77649.0, 80539.0, 82598.0, 83601.0, 84519.0, 85663.0, 86102.0, 88664.0] 

         secid      sdate      edate   permno  score
7         5007 2012-04-12 2016-08-17  13343.0      5
109       5111 2021-03-18 2023-02-02  20768.0      5
118       5121 2018-02-28 2019-08-13  17295.0      5
130       5131 2007-04-02 2024-05-02  88960.0      5
139       5139 2002-07-29 2024-12-31  89462.0      1
...        ...        ...    

In [9]:
goods = crsp_secid_link['permno'].isin(universe)
secids = crsp_secid_link.loc[goods,'secid'].unique()
secids

array([  5876,   6514, 100972, 101121, 101281, 101368, 102021, 102386,
       102548, 102660, 103022, 103106, 103125, 103157, 103313, 103355,
       103912, 103979, 103980, 104361, 104411, 104508, 104560, 104634,
       105174, 105340, 105683, 105696, 106125, 106329, 106566, 106638,
       106689, 106808, 107078, 107318, 107430, 107544, 107704, 108055,
       108117, 108130, 108893, 108910, 109224, 110169, 110337, 110513,
       110952, 111337, 111436, 112136, 124322, 178248, 182631, 208207,
       208583, 210407])

In [10]:
print('secids:',secids.shape)
print('permnos:',len(universe))

secids: (58,)
permnos: 50


Seems we have a problem: some permnos have multiple secids, probably secid changes due to corporate actions while permno stays the same. We can me more precise. 

In [11]:
# Merge SPX composition with secid link based on overlapping date ranges
# For each permno in comp, find the corresponding secid(s) where date ranges overlap
def merge_comp_with_secid(comp_df, link_df):
    """
    Merge SPX composition with secid link based on overlapping date ranges.
    
    Parameters
    ----------
    comp_df : DataFrame
        SPX composition with columns: permno, start, ending
    link_df : DataFrame
        Link table with columns: permno, secid, sdate, edate, score

    Returns
    -------
    comp_with_secid : DataFrame
        Columns:
        - permno, start, ending             (from comp_df)
        - secid                             (chosen best link)
        - sdate, edate                      (link validity from link_df)
        - overlap_start, overlap_end        (intersection of [start, ending] and [sdate, edate])
    """
    merged_rows = []

    for _, comp_row in comp_df.iterrows():
        permno = comp_row['permno']
        comp_start = comp_row['start']
        comp_end = comp_row['ending']

        # candidate links with overlapping validity window
        link_matches = link_df[
            (link_df['permno'] == permno) &
            (link_df['sdate'] <= comp_end) &
            (link_df['edate'] >= comp_start)
        ].copy()

        if len(link_matches) > 0:
            # compute overlap interval with SPX membership
            link_matches['overlap_start'] = link_matches['sdate'].apply(
                lambda x: max(comp_start, x)
            )
            link_matches['overlap_end'] = link_matches['edate'].apply(
                lambda x: min(comp_end, x)
            )
            link_matches['overlap_days'] = (
                link_matches['overlap_end'] - link_matches['overlap_start']
            ).dt.days

            # choose best: lowest score, then longest overlap
            link_matches = link_matches.sort_values(
                ['score', 'overlap_days'],
                ascending=[True, False]
            )
            best_match = link_matches.iloc[0]

            merged_row = comp_row.copy()
            merged_row['secid'] = best_match['secid']
            merged_row['sdate'] = best_match['sdate']
            merged_row['edate'] = best_match['edate']
            merged_row['overlap_start'] = best_match['overlap_start']
            merged_row['overlap_end'] = best_match['overlap_end']
            merged_rows.append(merged_row)
        else:
            merged_row = comp_row.copy()
            merged_row['secid'] = np.nan
            merged_row['sdate'] = pd.NaT
            merged_row['edate'] = pd.NaT
            merged_row['overlap_start'] = pd.NaT
            merged_row['overlap_end'] = pd.NaT
            merged_rows.append(merged_row)

    comp_with_secid = pd.DataFrame(merged_rows)
    return comp_with_secid

In [12]:
# Perform the merge
comp_with_secid = merge_comp_with_secid(comp, crsp_secid_link)

# Display results
print("SPX composition with secid:")
print(comp_with_secid.head(20))
print(f"\nTotal rows: {len(comp_with_secid)}")
print(f"Rows with secid: {comp_with_secid['secid'].notna().sum()}")
print(f"Rows without secid: {comp_with_secid['secid'].isna().sum()}")

SPX composition with secid:
     permno      start     ending     secid      sdate      edate  \
6     10104 1989-08-03 2024-12-31  108505.0 1996-01-01 2024-12-31   
7     10107 1994-06-07 2024-12-31  107525.0 1996-01-02 2024-12-31   
11    10138 1999-10-13 2024-12-31  109181.0 1996-01-01 2024-12-31   
12    10145 1925-12-31 2024-12-31  105785.0 1996-01-01 2024-12-31   
13    10147 1996-03-28 2016-09-07  104049.0 1996-01-02 2016-09-06   
20    10225 1925-12-31 2014-04-30  104958.0 1996-01-01 2014-04-30   
28    10299 2000-04-03 2017-03-10  106982.0 1996-01-02 2017-03-10   
45    10516 1981-07-30 2024-12-31  101639.0 1996-01-02 2024-12-31   
54    10696 2001-04-02 2024-12-31  104870.0 1996-01-02 2024-12-31   
72    10909 2010-04-30 2022-06-07  102893.0 1996-01-02 2022-06-07   
92    11308 1957-03-01 2024-12-31  103125.0 1996-01-02 2024-12-31   
101   11403 2017-09-18 2024-12-31  102602.0 1996-01-02 2024-12-31   
102   11404 1925-12-31 2024-12-31  103355.0 1996-01-01 2024-12-31   
108   

In [13]:
sel = comp_with_secid['permno'].isin(universe)
print(comp_with_secid[sel].sort_values(by = 'permno'))
secids = tuple(comp_with_secid.loc[sel,'secid'].drop_duplicates().astype(str))
print('secids:',len(secids))
print('permnos:',len(universe))


      permno      start     ending     secid      sdate      edate  \
92     11308 1957-03-01 2024-12-31  103125.0 1996-01-02 2024-12-31   
102    11404 1925-12-31 2024-12-31  103355.0 1996-01-01 2024-12-31   
125    11786 2018-03-19 2023-03-14  110169.0 1996-01-02 2023-03-10   
272    13856 1957-03-01 2024-12-31  108893.0 1996-01-02 2024-12-31   
427    16432 1925-12-31 2019-02-26  105340.0 1996-01-02 2024-12-31   
433    16600 1957-03-01 2024-12-31  105696.0 1996-01-02 2024-12-31   
460    17144 1969-02-13 2024-12-31  105174.0 1996-01-02 2024-12-31   
493    17750 1957-03-01 2024-12-31  106689.0 1996-01-02 2024-12-31   
513    18163 1957-03-01 2024-12-31  109224.0 1996-01-02 2024-12-31   
527    18411 1944-06-07 2024-12-31  110337.0 1996-01-02 2024-12-31   
547    18729 1957-03-01 2024-12-31  103157.0 1996-01-02 2024-12-31   
639    20482 1957-03-01 2024-12-31  100972.0 1996-01-01 2024-12-31   
668    21207 1969-05-15 2024-12-31  108130.0 1996-01-02 2024-12-31   
698    21776 1944-06

### Let us discuss what kind of predictors we can construct from options data!  
- Average volatility
- Skewness
- Any open interest or volume-based features? 

In [ ]:
db = wrds.Connection(wrds_username=Options.wrds_name)

In [106]:
sql = '''
select date, secid, AVG(impl_volatility) as iv 
from optionm.vsurfd%(year)s
where secid in %(secids)s and days = 30 and abs(delta)<=50 
group by date, secid
'''

params = {}
params['year']   = 2015
params['secids'] = tuple(secids[:10])
data = db.raw_sql(sql,params = params)

# convert the date to the date format --
data.loc[:,'date'] = pd.to_datetime(data.loc[:,'date'])

/var/folders/1g/3hly6g694k12jhfx7g3xkkj40000gq/T/ipykernel_89121/1481192122.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '<DatetimeArray>
['2015-04-01 00:00:00', '2015-03-30 00:00:00', '2015-11-04 00:00:00',
 '2015-05-04 00:00:00', '2015-11-09 00:00:00', '2015-03-11 00:00:00',
 '2015-10-28 00:00:00', '2015-08-21 00:00:00', '2015-03-17 00:00:00',
 '2015-07-01 00:00:00',
 ...
 '2015-08-05 00:00:00', '2015-10-28 00:00:00', '2015-08-12 00:00:00',
 '2015-05-28 00:00:00', '2015-12-24 00:00:00', '2015-01-28 00:00:00',
 '2015-05-27 00:00:00', '2015-12-28 00:00:00', '2015-09-29 00:00:00',
 '2015-03-18 00:00:00']
Length: 2520, dtype: datetime64[ns]' has dtype incompatible with string, please explicitly cast to a compatible dtype first.
  data.loc[:,'date'] = pd.to_datetime(data.loc[:,'date'])


In [107]:
data

,date,secid,iv
0,2015-04-01,103355.0,0.205473
1,2015-03-30,105340.0,0.308403
2,2015-11-04,110337.0,0.15908
3,2015-05-04,110337.0,0.153728
4,2015-11-09,110169.0,0.312001
...,...,...,...
2515,2015-01-28,106689.0,0.173098
2516,2015-05-27,105174.0,0.160419
2517,2015-12-28,103355.0,0.182993
2518,2015-09-29,110337.0,0.204215


Let us try to program some additional features! 

In [108]:
UNIVERSE_SECID = f'AND secid in {secids[:10]}'
YR = 2020
# a more complex example that we use to compute ATM IV and some skewness proxies
sql_wrds = f'''
            select a.date, a.secid,
            a.iv_atm as iv_atm,
            b.iv_otmput-a.iv_atm as put_skew,
            (b.iv_otmput-a.iv_atm)/a.iv_atm as put_skew_rel,
            c.iv_otmcall-a.iv_atm as call_skew,
            c.iv_otmcall-b.iv_otmput as skew from

            (select date, secid, AVG(impl_volatility) as iv_atm
            from optionm.vsurfd{YR}
            where days = 30 and abs(delta)=50
            {UNIVERSE_SECID}
            group by date, secid) as a,

            (select date, secid, AVG(impl_volatility) as iv_otmput
            from optionm.vsurfd{YR}
            where days = 30 and delta >=-20 and delta<=-10 
            {UNIVERSE_SECID}
            group by date, secid) as b,
            
            (select date, secid, AVG(impl_volatility) as iv_otmcall
            from optionm.vsurfd{YR}
            where days = 30 and delta >=10 and delta <=20
            {UNIVERSE_SECID}
            group by date, secid) as c

            where a.date= b.date and a.secid = b.secid
            and a.date = c.date and a.secid = c.secid
            '''
data_om = db.raw_sql(sql_wrds)

In [110]:
data_om.set_index(['secid','date'], inplace=True)
data_om.sort_index(inplace=True)
data_om.corr()
# Function definition to download all the data -> save

,iv_atm,put_skew,put_skew_rel,call_skew,skew
iv_atm,1.000000,0.725758,0.180898,0.072442,-0.783078
put_skew,0.725758,1.000000,0.703848,0.491835,-0.823643
put_skew_rel,0.180898,0.703848,1.000000,0.598087,-0.415642
call_skew,0.072442,0.491835,0.598087,1.000000,0.088679
skew,-0.783078,-0.823643,-0.415642,0.088679,1.000000


# Function definition to download all the data -> save

Let us pack everything we programmed in a couple of functions, so that we can run them anytime. 
We need to compute some features from the options data and save them in a file. Let us compute average OTM volatility, and some definitions of the skew/ smirk.

In [111]:
comp = pd.read_csv(FileNames.fn_sp500comp)
comp['start']  = pd.to_datetime(comp['start'])
comp['ending'] = pd.to_datetime(comp['ending'])
comp = comp.loc[comp.ending>DATA_FROM_LIMIT,:] 

universe_permno = tuple(pd.read_csv(FileNames.fn_universe)['permno'])

crsp_secid_link = pd.read_csv(FileNames.fn_id_link)
crsp_secid_link.columns = [z.lower().strip() for z in crsp_secid_link.columns]
crsp_secid_link['sdate'] = pd.to_datetime(crsp_secid_link['sdate'])
crsp_secid_link['edate'] = pd.to_datetime(crsp_secid_link['edate'])
crsp_secid_link = crsp_secid_link.loc[crsp_secid_link['edate']>DATA_FROM_LIMIT,:]

comp_with_secid = merge_comp_with_secid(comp, crsp_secid_link)

goods = comp_with_secid.permno.isin(universe_permno)
universe_secid = tuple(comp_with_secid.loc[goods,'secid'].dropna().drop_duplicates())

UNIVERSE_PERMNO = f'AND a.permno in {universe_permno}'
UNIVERSE_SECID = f'AND secid in {universe_secid}'

UNIVERSE_SECID
# UNIVERSE_PERMNO

'AND secid in (103125.0, 103355.0, 110169.0, 108893.0, 105340.0, 105696.0, 105174.0, 106689.0, 109224.0, 110337.0, 103157.0, 100972.0, 108130.0, 104508.0, 106566.0, 107430.0, 104361.0, 101368.0, 104560.0, 102660.0, 106638.0, 103979.0, 108910.0, 107318.0, 101281.0, 103106.0, 106808.0, 110952.0, 104411.0, 106329.0, 107544.0, 103313.0, 101121.0, 103912.0, 107704.0, 103980.0, 111337.0, 110513.0, 106125.0, 108055.0, 111436.0, 105683.0)'

In [113]:
len(UNIVERSE_SECID)
len(UNIVERSE_PERMNO)

466

In [ ]:
db = wrds.Connection(wrds_username=Options.wrds_name)
UNIVERSE_SECID = f'AND secid in {universe_secid[:10]}'
YR = 2020
# a more complex example that we use to compute ATM IV and some skewness proxies
sql_wrds = f'''
            select a.date, a.secid,
            a.iv_atm as iv_atm,
            b.iv_otmput-a.iv_atm as put_skew,
            c.iv_otmcall-a.iv_atm as call_skew,
            c.iv_otmcall-b.iv_otmput as skew from

            (select date, secid, AVG(impl_volatility) as iv_atm
            from optionm.vsurfd{YR}
            where days = 30 and abs(delta)=50
            {UNIVERSE_SECID}
            group by date, secid) as a,

            (select date, secid, AVG(impl_volatility) as iv_otmput
            from optionm.vsurfd{YR}
            where days = 30 and delta >=-20 and delta<=-10 
            {UNIVERSE_SECID}
            group by date, secid) as b,
            
            (select date, secid, AVG(impl_volatility) as iv_otmcall
            from optionm.vsurfd{YR}
            where days = 30 and delta >=10 and delta <=20
            {UNIVERSE_SECID}
            group by date, secid) as c

            where a.date= b.date and a.secid = b.secid
            and a.date = c.date and a.secid = c.secid
            '''
data_om = db.raw_sql(sql_wrds)

In [ ]:
data_om.head()

In [ ]:
# here we merge the data with permno to be able to merge later with the other tables
# now we use the overlap_start/overlap_end to enforce date validity of the link
data_om = data_om.merge(
    comp_with_secid[['permno', 'secid', 'overlap_start', 'overlap_end']],
    how='left',
    on='secid'
)

# keep only rows where the option data date lies within the valid link interval
data_om = data_om[
    (data_om['date'] >= data_om['overlap_start']) &
    (data_om['date'] <= data_om['overlap_end'])
].copy()

data_om

In [ ]:
def prepare_save_om(load_om_data = True, 
                    om_from = 2014, 
                    om_to = 2024,
                    restrict_universe = True): # the universe is restricted to the SPX composition    

    UNIVERSE_PERMNO = ''
    UNIVERSE_SECID = ''

    comp = pd.read_csv(FileNames.fn_sp500comp)
    comp['start']  = pd.to_datetime(comp['start'])
    comp['ending'] = pd.to_datetime(comp['ending'])
    comp = comp.loc[comp.ending>DATA_FROM_LIMIT,:] 

    crsp_secid_link = pd.read_csv(FileNames.fn_id_link)
    crsp_secid_link.columns = [z.lower().strip() for z in crsp_secid_link.columns]
    crsp_secid_link['sdate'] = pd.to_datetime(crsp_secid_link['sdate'])
    crsp_secid_link['edate'] = pd.to_datetime(crsp_secid_link['edate'])
    crsp_secid_link = crsp_secid_link.loc[crsp_secid_link['edate']>DATA_FROM_LIMIT,:]

    if restrict_universe:
        comp_with_secid = merge_comp_with_secid(comp, crsp_secid_link)
        universe_secid = tuple(comp_with_secid.loc[:,'secid'].dropna().drop_duplicates())
        UNIVERSE_PERMNO = f'AND a.permno in {universe_permno}'
        UNIVERSE_SECID = f'AND secid in {universe_secid}'
    else:
        UNIVERSE_PERMNO = f''
        UNIVERSE_SECID = f''

    
    if load_om_data:
        
        db = wrds.Connection(wrds_username=Options.wrds_name)
        
        start = time.time()
        
        data_om = pd.DataFrame()
        for YR in range(om_from, om_to+1):
        
            sql_wrds = f'''
            select a.date, a.secid,
            a.iv_atm as iv_atm,
            b.iv_otmput-a.iv_atm as put_skew,
            c.iv_otmcall-a.iv_atm as call_skew,
            (b.iv_otmput-a.iv_atm)/a.iv_atm as put_skew_rel,
            (c.iv_otmcall-a.iv_atm)/a.iv_atm as call_skew_rel from

            (select date, secid, AVG(impl_volatility) as iv_atm
            from optionm.vsurfd{YR}
            where days = 30 and abs(delta)=50
            {UNIVERSE_SECID}
            group by date, secid) as a,

            (select date, secid, AVG(impl_volatility) as iv_otmput
            from optionm.vsurfd{YR}
            where days = 30 and delta >=-20 and delta<=-10 
            {UNIVERSE_SECID}
            group by date, secid) as b,
            
            (select date, secid, AVG(impl_volatility) as iv_otmcall
            from optionm.vsurfd{YR}
            where days = 30 and delta >=10 and delta <=20
            {UNIVERSE_SECID}
            group by date, secid) as c

            where a.date= b.date and a.secid = b.secid
            and a.date = c.date and a.secid = c.secid
            '''
            try:
                data_om = pd.concat([data_om, db.raw_sql(sql_wrds)], axis = 0)
                print(f'finished downloading OM {YR} in {(time.time()-start)/60:.2f} minutes')
                pass
            except: 
                pass
            
        db.close()
        data_om['date'] = pd.to_datetime(data_om['date'])
        
        # merge with PERMNO using secid AND enforce link validity via overlap_start/overlap_end
        data_om = data_om.merge(
            comp_with_secid[['permno', 'secid', 'overlap_start', 'overlap_end']],
            how='left',
            on='secid'
        )

        # keep only observations where the option date lies within the valid link interval
        # data_om = data_om[
        #     (data_om['date'] >= data_om['overlap_start']) &
        #     (data_om['date'] <= data_om['overlap_end'])
        # ].copy()

        # drop helper columns if not needed in final features
        data_om = data_om.drop(columns=['overlap_start', 'overlap_end'])

        # keep only rows with a valid permno and set index
        data_om = data_om.dropna(subset=['permno'])
        data_om = data_om.set_index(['permno', 'date']).sort_index()
        data_om = data_om[~data_om.index.duplicated()]

        data_om.to_parquet(FileNames.fn_option_features)
        print(f'Saved OM data')

    return 0

In [45]:
prepare_save_om(load_om_data = True, 
                    om_from = 2014, 
                    om_to = 2024,
                    restrict_universe = True) # the universe is restricted to the SPX composition

Loading library list...
Done
finished downloading OM 2014 in 0.74 minutes
finished downloading OM 2015 in 1.46 minutes
finished downloading OM 2016 in 2.45 minutes
finished downloading OM 2017 in 3.24 minutes
finished downloading OM 2018 in 4.02 minutes
finished downloading OM 2019 in 4.64 minutes
finished downloading OM 2020 in 5.39 minutes
finished downloading OM 2021 in 6.49 minutes
finished downloading OM 2022 in 7.23 minutes
finished downloading OM 2023 in 8.07 minutes
finished downloading OM 2024 in 8.86 minutes
Saved OM data


0

In [4]:
# adjust the code to load the features adding the ones from option metrics
def load_features(crsp = True,
                 om = True, 
                  merge = True, 
                  cols_to_drop = ['overlap_start','overlap_end', 'secid']):
    features_crsp, features_om = pd.DataFrame(), pd.DataFrame()
    if crsp: 
        features_crsp = pd.read_parquet(FileNames.fn_stock_features_labels)
        features_crsp = features_crsp.sort_index()
        pass
    if om:
        features_om = pd.read_parquet(FileNames.fn_option_features)
        features_om = features_om.sort_index()
    if merge:
        res = pd.concat([features_crsp, features_om], axis = 1).sort_index()
        cols_to_drop = [z for z in cols_to_drop if z in res.columns]
        res = res.drop(columns = cols_to_drop)
    else:
        res = (features_crsp, features_om)
    return res

    feat = load_features(merge=True)

In [5]:
feat = load_features(merge=True)
feat.dropna(subset = ['ret'])

ret          mktcap    fret1d    fret5d    mom12m  \
permno date                                                                 
10104  2014-12-31 -0.008161    200915.32336 -0.014232 -0.032033  0.122049   
       2015-01-02 -0.014232    197479.77399 -0.013986 -0.018511  0.113319   
       2015-01-05 -0.013986    194669.29911 -0.010324 -0.007343  0.115964   
       2015-01-06 -0.010324    191419.68753  0.000232 -0.004869  0.112789   
       2015-01-07  0.000232    189443.57238  0.006025 -0.002087  0.108291   
...                     ...             ...       ...       ...       ...   
93436  2024-12-24  0.073572  1382251.868101  -0.01763       NaN  0.408132   
       2024-12-26  -0.01763    1483946.5368 -0.049479       NaN  0.505321   
       2024-12-27 -0.049479    1457784.5478 -0.033012       NaN  0.438060   
       2024-12-30 -0.033012    1385654.4996  -0.03251       NaN  0.432697   
       2024-12-31  -0.03251    1339911.1446      <NA>       NaN  0.349266   

                      mom6m     rev1m  beta_mktrf  beta_smb  beta_hml  \
permno date                                                             
10104  2014-12-31  0.016560  0.068678    1.141866 -0.269630 -0.541900   
       2015-01-02  0.012509  0.050972    1.140102 -0.258900 -0.539181   
       2015-01-05  0.017593  0.039230    1.123524 -0.252731 -0.542466   
       2015-01-06  0.005057  0.032675    1.124913 -0.251336 -0.540608   
       2015-01-07 -0.011450  0.031929    1.116593 -0.245790 -0.519419   
...                     ...       ...         ...       ...       ...   
93436  2024-12-24  0.954884  0.311209    2.617136 -0.094096 -0.526324   
       2024-12-26  0.966980  0.341239    2.628032 -0.115337 -0.520952   
       2024-12-27  0.915650  0.276233    2.638371 -0.107069 -0.528668   
       2024-12-30  0.919696  0.253898    2.651733 -0.125228 -0.531348   
       2024-12-31  0.861911  0.170009    2.649544 -0.127412 -0.541994   

                   beta_mom  idvar_ff4    iv_atm  put_skew  call_skew  \
permno date                                                             
10104  2014-12-31 -0.223871   0.024360  0.178476  0.055114   0.004481   
       2015-01-02 -0.221639   0.024561  0.182134  0.062998  -0.002216   
       2015-01-05 -0.200264   0.024600  0.199347  0.058988   0.003852   
       2015-01-06 -0.201985   0.024602  0.207479  0.055371  -0.002126   
       2015-01-07 -0.212445   0.024808  0.199981    0.0594  -0.003771   
...                     ...        ...       ...       ...        ...   
93436  2024-12-24 -1.043935   0.300170  0.690471  0.039022   0.060435   
       2024-12-26 -1.049314   0.300543   0.71662  0.030415   0.064651   
       2024-12-27 -1.038589   0.301041  0.701358  0.055148   0.064214   
       2024-12-30 -1.063133   0.299563  0.706274  0.040682   0.065339   
       2024-12-31 -1.044894   0.300308  0.697715   0.04189   0.063961   

                   put_skew_rel  call_skew_rel  
permno date                                     
10104  2014-12-31      0.308803       0.025105  
       2015-01-02      0.345892      -0.012168  
       2015-01-05      0.295904       0.019325  
       2015-01-06      0.266873      -0.010244  
       2015-01-07       0.29703      -0.018856  
...                         ...            ...  
93436  2024-12-24      0.056515       0.087527  
       2024-12-26      0.042442       0.090217  
       2024-12-27       0.07863       0.091557  
       2024-12-30      0.057601       0.092513  
       2024-12-31      0.060039       0.091673  

[1590520 rows x 17 columns]

## Some snippets for the class

In [ ]:
# by observing some weird characteristic values you might want to winsorize them
# from scipy.stats.mstats import winsorize
#
# # vars2wins = ['iv']
# # for z in vars2wins:
# #     goods    = data[z].notna()
# #     data.loc[goods,z] = winsorize(data.loc[goods,z], limits=0.01).data
# # data.describe()
#
#
# def winsorize_all(data, threshhold):
#     '''
#     Function winsorizing all columns in a data frame
#     '''
#     vars2wins = data.columns
#     for z in vars2wins:
#         goods = data[z].notna()
#         if not goods.sum() == 0:  # check that all observation in column is not missing
#             data.loc[goods, z] = winsorize(data.loc[goods, z], limits=threshhold).data
#     return data
#
#
# # winsorize by period
# cols_winso = list(f2.columns)
# f2[cols_winso] = f2.groupby(pd.Grouper(level = 'date',freq = 'Y'))[cols_winso].apply(winsorize_all, threshhold=0.01)

In [ ]:
feat.describe(percentiles=[0.01,0.99]).round(3)